<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_07_model_analysis/seq2one/stage_07_01a_mlp_seq2one_robustness_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_01a -  SEQ2ONE - Modelo MLP - Robustness Testing**

En esta sección iniciamos el proceso de **tuning del modelo MLP bajo el enfoque many-to-one (SEQ2ONE)**, utilizando como variable objetivo el **`delta_60`**, es decir, la variación en puntos del MNQ en los próximos 60 minutos.

El enfoque SEQ2ONE consiste en utilizar una ventana histórica de tamaño \( L \) minutos como entrada y predecir un único valor futuro correspondiente al horizonte seleccionado. En este caso:

$$
X \in \mathbb{R}^{(L \times F)} \quad \longrightarrow \quad y \in \mathbb{R}
$$

donde:

- $L$ = window size  
- $F$ = número de features (36)
- $y$ = `delta_60`  

---

**Motivación**

De acuerdo con el workflow de investigación y modelado presentado en el libro *Machine Learning for Algorithmic Trading*, el diseño y tuning del modelo corresponde a la etapa de:

> **Design, tune, and evaluate ML models to generate trading signals**

En esta fase, el objetivo no es todavía el backtesting completo, sino encontrar una configuración que:

- Generalice correctamente (early stopping sobre VALID)  
- Maximice capacidad predictiva (R², RMSE, MAE)  
- Mantenga estabilidad direccional (DA)  

---

**Objetivo del tuning**

El propósito del tuning será:

1. Evaluar distintos tamaños de ventana $L$.  
2. Ajustar hiperparámetros del MLP:
   - `hidden_dim`
   - `dropout`
   - `learning_rate`
   - `weight_decay`
3. Seleccionar la mejor configuración usando exclusivamente el conjunto **VALID**.  
4. Reservar el conjunto **TEST** para evaluación final no sesgada.  

Este procedimiento evita leakage y respeta el principio de generalización fuera de muestra, fundamental en modelado financiero.

---

**Contexto técnico**

El MLP:

- Recibe ventanas aplanadas en formato 2D.  
- No modela memoria temporal explícita (a diferencia de LSTM/GRU).  
- Aprende relaciones no lineales entre patrones recientes del mercado y el `delta_60`.  

Por lo tanto, el tamaño de ventana seleccionado determinará indirectamente cuánta información temporal puede capturar el modelo.

---

En las siguientes secciones se definirá el espacio de búsqueda y se ejecutará el proceso de tuning controlado sobre TRAIN/VALID.



# **BLOQUE DE EJECUCIÓN COMPLETO**

In [1]:
window_sizes = [90, 180]
targets = ['delta_60']
splits = ['train', 'valid', 'test']

## **1. Imports + paths**

In [2]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


## **3. Rutas de ventanas seq2one y scalers**

In [4]:
from pathlib import Path
import os

WINDOWS_SEQ2ONE_DIR = Path(
    os.environ.get("WINDOWS_SEQ2ONE_DIR", "data/windows/seq2one/")
)

SCALERS_DIR = Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

In [5]:
windows_paths = {}

for w in window_sizes:
    windows_paths[w] = {}

    for t in targets:
        windows_paths[w][t] = {}

        for s in splits:
            path = (
                DRIVE_DIR
                / WINDOWS_SEQ2ONE_DIR
                / f"L{w}"
                / f"windows_{t}_{s}.npz"
            )

            windows_paths[w][t][s] = path

#display(windows_paths)

#Como llamarlo:
#path_train_L60_delta = windows_paths[60]['delta_90']['train']
#print(path_train_L60_delta)

In [6]:
scalers_paths = {}
for t in targets:
  scalers_paths[t] = {}
  path = (
                DRIVE_DIR
                / SCALERS_DIR
                / f"scaler_{t}.pkl"
            )

  scalers_paths[t] = path

display(scalers_paths)

{'delta_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl')}

## **4. Reproducibilidad**

In [7]:
#def set_seeds(seed: int = 42) -> None:
#    random.seed(seed)
#    np.random.seed(seed)
#    os.environ["PYTHONHASHSEED"] = str(seed)
#
#set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [8]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [9]:
#print(compute_seq2one_metrics.__doc__)

## **6. Carga de data windows**

In [10]:
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.
    """

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Usar contexto para cerrar correctamente el archivo
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # Opcional pero recomendable: copiar a memoria
        X = X.copy()
        y = y.copy()

    return X, y


In [11]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [12]:
from typing import Any, Dict, Mapping
from pathlib import Path

def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scalers_path: Mapping[str, Path],
    splits: tuple[str, ...] = ("train", "valid", "test"),
) -> Dict[str, Any]:
    """
    Carga X/y para los splits solicitados y el scaler correspondiente
    a un (window_size, target).

    windows_paths[L][target][split] -> Path (.npz con X,y)
    scalers_path[target] -> Path (scaler)
    """

    # --------------------------
    # 1) Validaciones base
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    if target not in scalers_path:
        raise KeyError(f"target='{target}' no existe en scalers_path")

    valid_splits = {"train", "valid", "test"}
    splits_set = set(splits)
    unknown = splits_set - valid_splits
    if unknown:
        raise ValueError(f"splits inválidos: {sorted(unknown)}. Usar {sorted(valid_splits)}")

    # Validar que existan los splits solicitados para ese target
    available = set(windows_paths[window_size][target].keys())
    missing = splits_set - available
    if missing:
        raise KeyError(
            f"Faltan splits {sorted(missing)} en windows_paths[{window_size}]['{target}']. "
            f"Disponibles: {sorted(available)}"
        )

    # --------------------------
    # 2) Paths (solo los necesarios)
    # --------------------------
    split_paths: Dict[str, Path] = {sp: windows_paths[window_size][target][sp] for sp in splits}
    scaler_path = scalers_path[target]

    # --------------------------
    # 3) Carga por split
    # --------------------------
    out_splits: Dict[str, Dict[str, Any]] = {}
    out_paths: Dict[str, str] = {}

    for sp, p in split_paths.items():
        X, y = load_npz_windows(p)
        out_splits[sp] = {"X": X, "y": y}
        out_paths[sp] = str(p)

    scaler = load_scaler(scaler_path)
    out_paths["scaler"] = str(scaler_path)

    # --------------------------
    # 4) Inferir horizonte (robusto)
    # --------------------------
    try:
        horizon = int(target.split("_")[-1])
    except Exception as e:
        raise ValueError(f"No se pudo inferir horizon desde target='{target}'. Esperado sufijo '_<int>'") from e

    # --------------------------
    # 5) Retorno
    # --------------------------
    out: Dict[str, Any] = {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": out_paths,
        "scaler": scaler,
    }
    out.update(out_splits)

    return out

In [13]:
import numpy as np

def maybe_flatten_X(X: np.ndarray, *, flatten: bool) -> np.ndarray:
    """
    Si flatten=True y X es 3D (N,L,F) -> (N, L*F)
    Si flatten=False -> retorna X tal cual.
    """
    if not flatten:
        return X
    if X.ndim == 3:
        N, L, F = X.shape
        return X.reshape(N, L * F)
    if X.ndim == 2:
        return X
    raise ValueError(f"X debe ser 2D o 3D, recibí shape={X.shape}")

In [14]:
def create_bundles(
    window_size,
    targets: list,
    windows_paths=windows_paths,
    scalers_paths=scalers_paths,
    *,
    flatten_X: bool = False,
    splits: tuple[str, ...] = ("train", "valid", "test"),
    verbose_shapes: bool = True,
    ):
    bundles = []
    for t in targets:
        b = load_windows_and_scaler(
            window_size=window_size,
            target=t,
            windows_paths=windows_paths,
            scalers_path=scalers_paths,
            splits=splits,
        )

        if flatten_X:
            for sp in splits:
                b[sp]["X"] = maybe_flatten_X(b[sp]["X"], flatten=True)

        bundles.append(b)

    if verbose_shapes:
        for b in bundles:
            for sp in splits:
                print(f"H{b['horizon']} {sp.capitalize():<5}:", b[sp]["X"].shape, b[sp]["y"].shape)
            print(f"Scaler H{b['horizon']}:", type(b["scaler"]).__name__)

    return tuple(bundles)

NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **7. Sanity Check**

In [15]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [16]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [17]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

## **8. Métricas ML**

In [18]:
import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    horizon: int,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])

In [19]:
def get_metrics_torch(bundle, model, *, device) -> tuple[dict, dict]:
  # -------- VALID --------
  X_valid = bundle["valid"]["X"]
  y_valid = bundle["valid"]["y"]
  y_pred_valid = predict_mlp(model, X_valid, device=device)
  metrics_valid = compute_seq2one_metrics(y_valid, y_pred_valid, compute_r2=True)

  # -------- TEST --------
  X_test = bundle["test"]["X"]
  y_test = bundle["test"]["y"]
  y_pred_test = predict_mlp(model, X_test, device=device)
  metrics_test  = compute_seq2one_metrics(y_test, y_pred_test,  compute_r2=True)

  return metrics_valid, metrics_test

## **9. Gestión de dataset de métricas**

In [20]:
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [21]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"seq2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

# **DEFINICIÓN DE MODELO**

## **11. Definición del modelo — placeholder**

### **11.1. Imports (PyTorch) + semillas**

In [22]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [23]:
import os
import random
import numpy as np
import torch

def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # determinismo (puede bajar performance)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # fuerza algoritmos deterministas (puede fallar si alguna op no tiene versión determinista)
    torch.use_deterministic_algorithms(True)

    # opcional, ayuda a determinismo en algunas matmuls CUDA (si usa GPU)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# NO lo llame aquí si luego hará loop de seeds; llámelo dentro del loop.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


### **11.2. Dataset/DataLoader desde bundle**

In [24]:
import random
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

def _seed_worker(worker_id: int) -> None:
    # asegura que numpy/random en cada worker quede determinista
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loaders_from_bundle(
    bundle: dict,
    *,
    splits: tuple[str, ...] = ("train", "valid", "test"),
    batch_size: int = 4096,
    num_workers: int = 0,
    pin_memory: bool | None = None,
    seed: int | None = None,
) -> dict:
    if pin_memory is None:
        pin_memory = torch.cuda.is_available()

    loaders: dict[str, DataLoader] = {}

    # generator para controlar el shuffle de forma reproducible
    g = None
    if seed is not None:
        g = torch.Generator()
        g.manual_seed(seed)

    for split in splits:
        if split not in bundle:
            continue

        X = np.asarray(bundle[split]["X"], dtype=np.float32)

        y = np.asarray(bundle[split]["y"], dtype=np.float32)
        if y.ndim == 1:
            y = y.reshape(-1, 1)

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        shuffle = (split == "train")

        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin_memory,
            drop_last=False,
            generator=g if shuffle else None,
            worker_init_fn=_seed_worker if num_workers > 0 else None,
            persistent_workers=(num_workers > 0),
        )

    return loaders

### **11.3. Definición del modelo MLP (simple y controlado)**

In [25]:
import torch
import torch.nn as nn

# DEFINICIÓN DEL MODELO MLP (robusta)
class MLPSeq2One(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 128, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)

def build_mlp_from_bundle(
    bundle: dict,
    *,
    hidden_dim: int = 128,
    dropout: float = 0.0,
) -> MLPSeq2One:
    """
    Crea MLPSeq2One tomando in_dim desde bundle['train']['X'].
    Requiere que X esté 2D (flatten_X=True).
    """
    X = bundle["train"]["X"]
    if getattr(X, "ndim", None) != 2:
        raise ValueError(f"MLPSeq2One requiere X 2D (flatten_X=True). Recibido X.ndim={getattr(X,'ndim',None)}")
    in_dim = int(X.shape[1])
    return MLPSeq2One(in_dim=in_dim, hidden_dim=hidden_dim, dropout=dropout)

### **11.4. Entrenamiento con early stopping (VALID)**

In [26]:
import copy
import time
import torch
import torch.nn as nn
from typing import Optional
from torch.utils.data import DataLoader

def _ts():
    return time.strftime("%H:%M:%S")

@torch.no_grad()
def evaluate_mse(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    mse_sum = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        pred = model(xb)
        mse_sum += torch.sum((pred - yb) ** 2).item()
        n += yb.numel()
    return mse_sum / max(n, 1)

def train_mlp(
    loaders: dict,
    *,
    in_dim: Optional[int] = None,
    hidden_dim: int = 128,
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    min_delta: float = 1e-6,
    device: torch.device,
    verbose: bool = True,
    log_every: int = 0,
):
    if in_dim is None:
        xb0, _ = next(iter(loaders["train"]))
        in_dim = int(xb0.shape[1])

    if verbose:
        print(f"[{_ts()}] [TRAIN] START | in_dim={in_dim} hidden_dim={hidden_dim} dropout={dropout} "
              f"lr={lr} wd={weight_decay} max_epochs={max_epochs} patience={patience} min_delta={min_delta} "
              f"device={device.type}")

    t_global = time.perf_counter()

    model = MLPSeq2One(in_dim=in_dim, hidden_dim=hidden_dim, dropout=dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_state = None
    best_valid = float("inf")
    best_epoch = 0
    bad_epochs = 0

    epochs_ran = 0

    for epoch in range(1, max_epochs + 1):
        epochs_ran = epoch
        t_epoch = time.perf_counter()

        model.train()
        train_loss_sum = 0.0
        train_n = 0

        for b, (xb, yb) in enumerate(loaders["train"], start=1):
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

            bs = yb.numel()
            train_loss_sum += loss.item() * bs
            train_n += bs

            if verbose and log_every and (b % log_every == 0):
                print(f"[{_ts()}]   epoch={epoch:02d} batch={b:04d} | train_loss_avg={(train_loss_sum/max(train_n,1)):.6f}")

        train_loss_avg = train_loss_sum / max(train_n, 1)

        t0 = time.perf_counter()
        valid_mse = evaluate_mse(model, loaders["valid"], device)
        dt_valid = time.perf_counter() - t0
        dt_epoch = time.perf_counter() - t_epoch

        improved = valid_mse < (best_valid - min_delta)
        if improved:
            best_valid = valid_mse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if verbose:
            flag = "BEST" if improved else f"no_improve({bad_epochs}/{patience})"
            print(f"[{_ts()}] epoch={epoch:02d} | train_loss={train_loss_avg:.6f} | "
                  f"valid_mse={valid_mse:.6f} | {flag} | dt_valid={dt_valid:.2f}s | dt_epoch={dt_epoch:.2f}s")

        if bad_epochs >= patience:
            if verbose:
                print(f"[{_ts()}] [TRAIN] EARLY STOPPING | patience={patience} | best_valid_mse={best_valid:.6f} @epoch={best_epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    info = {
        "best_valid_mse": float(best_valid),
        "best_epoch": int(best_epoch),
        "epochs_ran": int(epochs_ran),
        "final_lr": float(opt.param_groups[0]["lr"]),
    }

    if verbose:
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [TRAIN] END | best_valid_mse={best_valid:.6f} @epoch={best_epoch} | dt_total={dt_all:.2f}s")

    return model, info

### **11.5. Predicciones MLP**


In [27]:
@torch.no_grad()
def predict_mlp(
    model,
    X: np.ndarray,
    *,
    device: torch.device,
    batch_size: int = 32768,
) -> np.ndarray:

    model.eval()

    X = np.asarray(X, dtype=np.float32)
    if X.ndim != 2:
        raise ValueError(f"MLPSeq2One requiere X 2D. Recibido X.ndim={X.ndim}")

    n = X.shape[0]
    preds = []

    for i in range(0, n, batch_size):
        xb = torch.from_numpy(X[i:i+batch_size]).to(device, non_blocking=True)
        yb = model(xb)

        if yb.ndim == 2 and yb.shape[1] == 1:
            yb = yb[:, 0]

        preds.append(yb.cpu().numpy())

    return np.concatenate(preds, axis=0)

## **12. Ejecución completa**

In [28]:
# ============================================================
# run_transformer (con loop por seeds) + run_transformer_incremental (skip por seeds)
# ============================================================

from __future__ import annotations

from pathlib import Path
from typing import Iterable, Any, Dict, Tuple

import gc
import time
import pandas as pd
import torch


def run_mlp(
    window_size: int,
    *,
    n_features: int = 36,
    hidden_dim: int = 128,
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    batch_size: int = 16384,
    max_epochs: int = 30,
    patience: int = 5,
    seeds: list[int] = (0, 1, 2, 3, 4),
    targets: list[str] = ("delta_60",),
    verbose: bool = True,
) -> pd.DataFrame:

    L = int(window_size)

    rows = []

    if verbose:
        print("\n" + "=" * 80)
        print(f"[{_ts()}] MLP | ROBUSTEZ SEEDS={list(seeds)} | WINDOW_SIZE=L{L} | in_dim={L*n_features} | "
              f"hd={hidden_dim} | dropout={dropout} | lr={lr} | w_decay={weight_decay}")
        print("=" * 80)

    t_global = time.perf_counter()

    for i, target in enumerate(targets, start=1):
        for j, seed in enumerate(seeds, start=1):
            t_run0 = time.perf_counter()

            # 0) SEED
            set_seed(seed)

            if verbose:
                print(f"[{_ts()}] [{i}/{len(targets)}] [{j}/{len(seeds)}] START target='{target}' | L{L} | seed={seed}")

            # 1) BUILD
            t0 = time.perf_counter()
            (bundle,) = create_bundles(
                window_size=L,
                targets=[target],
                windows_paths=windows_paths,
                scalers_paths=scalers_paths,
                flatten_X=True,
                splits=("train", "valid", "test"),
                verbose_shapes=False,
            )
            dt_build = time.perf_counter() - t0

            # 2) LOADERS (importante: pasar seed para shuffle reproducible)
            t0 = time.perf_counter()
            loaders = make_loaders_from_bundle(
                bundle,
                splits=("train", "valid", "test"),
                batch_size=batch_size,
                num_workers=0,
                seed=seed,  # <-- clave
            )
            dt_loaders = time.perf_counter() - t0

            # 3) TRAIN
            t0 = time.perf_counter()
            model, info = train_mlp(
                loaders,
                in_dim=L * n_features,
                hidden_dim=hidden_dim,
                dropout=dropout,
                lr=lr,
                weight_decay=weight_decay,
                max_epochs=max_epochs,
                patience=patience,
                device=device,
                verbose=verbose,
            )
            dt_train = time.perf_counter() - t0

            # 4) METRICS
            t0 = time.perf_counter()
            metrics_valid, metrics_test = get_metrics_torch(bundle, model, device=device)
            dt_eval = time.perf_counter() - t0

            # 5) DF (+ columnas de robustez)
            df_v = metrics_to_df(
                metrics_valid, model="mlp", split="valid",
                horizon=bundle["horizon"], window_size=bundle["window_size"], target=bundle["target"],
            )
            df_t = metrics_to_df(
                metrics_test, model="mlp", split="test",
                horizon=bundle["horizon"], window_size=bundle["window_size"], target=bundle["target"],
            )

            for df_ in (df_v, df_t):
                df_["seed"] = seed
                df_["best_valid_mse"] = info["best_valid_mse"]
                df_["best_epoch"] = info["best_epoch"]
                df_["epochs_ran"] = info["epochs_ran"]

            rows.append(df_v)
            rows.append(df_t)

            # cleanup
            del bundle, loaders, model, metrics_valid, metrics_test
            gc.collect()

            if verbose:
                dt_run = time.perf_counter() - t_run0
                print(f"[{_ts()}] DONE target='{target}' | L{L} | seed={seed} | "
                      f"dt_total={dt_run:.2f}s | build={dt_build:.2f}s | loaders={dt_loaders:.2f}s | "
                      f"train={dt_train:.2f}s | eval={dt_eval:.2f}s")

    df = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "seed", "split", "horizon_min", "model"])
          .reset_index(drop=True)
    )

    if verbose:
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [DONE] L{L} | rows={len(df)} | dt_total={dt_all:.2f}s")

    return df

In [29]:
# RUN MLP INCREMENTAL (ROBUSTEZ POR SEMILLAS)
import time
import pandas as pd

def _ts() -> str:
    return time.strftime("%H:%M:%S")


def run_mlp_incremental(
    window_sizes: list[int],
    *,
    n_features: int = 36,
    hidden_dim: int = 128,
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    batch_size: int = 16384,
    max_epochs: int = 30,
    patience: int = 5,
    seeds: list[int] = (0, 1, 2, 3, 4),
    targets: list[str] = ("delta_60",),  # mantener consistente con run_mlp
    name: str = "mlp",  # -> seq2one_{name}_metrics.parquet
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_testing",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Incremental para MLP con robustez por semillas.

    Requiere que `run_mlp(...)`:
      - acepte `seeds=` y `targets=`
      - devuelva un DF con columnas mínimas:
        ['model','split','window_size','target','horizon_min','MAE','RMSE','R2','DA','seed']
      - opcionalmente incluya:
        ['best_valid_mse','best_epoch','epochs_ran']

    Logs:
      [HH:MM:SS] [LOAD] ...
      [HH:MM:SS] [RUN] ...
      [HH:MM:SS] [DONE] ...
    """
    t_total0 = time.perf_counter()

    # 1) LOAD
    t0 = time.perf_counter()
    if verbose:
        print(f"[{_ts()}] [LOAD] Leyendo métricas existentes (name='{name}') ...")
    df_all = load_seq2one_metrics_if_exists(name=name, base_dir=base_dir)
    dt_load = time.perf_counter() - t0
    if verbose:
        print(f"[{_ts()}] [LOAD] OK | rows={len(df_all)} | dt={dt_load:.2f}s")

    # 2) Schema base (incluye seed + campos de training info)
    base_cols = [
        "model", "split", "window_size", "target", "horizon_min", "seed",
        "MAE", "RMSE", "R2", "DA",
        "best_valid_mse", "best_epoch", "epochs_ran",
        "hidden_dim", "dropout", "lr", "w_decay", "batch_size", "max_epochs", "patience",
    ]
    if df_all.empty:
        df_all = pd.DataFrame(columns=base_cols)

    # asegurar columnas (si el parquet viejo no las tenía)
    ensure_cols = {
        "seed": pd.NA,
        "best_valid_mse": pd.NA,
        "best_epoch": pd.NA,
        "epochs_ran": pd.NA,
        "hidden_dim": pd.NA,
        "dropout": pd.NA,
        "lr": pd.NA,
        "w_decay": pd.NA,
        "batch_size": pd.NA,
        "max_epochs": pd.NA,
        "patience": pd.NA,
    }
    for c, v in ensure_cols.items():
        if c not in df_all.columns:
            df_all[c] = v

    key_cols = [
        "model", "hidden_dim", "dropout", "lr", "w_decay", "batch_size", "max_epochs", "patience",
        "window_size", "target", "seed", "split", "horizon_min",
    ]

    # 3) NORM
    t0 = time.perf_counter()
    if len(df_all):
        df_all["window_size"] = pd.to_numeric(df_all["window_size"], errors="coerce").astype("Int64")
        df_all["horizon_min"] = pd.to_numeric(df_all["horizon_min"], errors="coerce").astype("Int64")
        df_all["seed"] = pd.to_numeric(df_all["seed"], errors="coerce").astype("Int64")

        df_all["hidden_dim"] = pd.to_numeric(df_all["hidden_dim"], errors="coerce").astype("Int64")
        df_all["batch_size"] = pd.to_numeric(df_all["batch_size"], errors="coerce").astype("Int64")
        df_all["max_epochs"] = pd.to_numeric(df_all["max_epochs"], errors="coerce").astype("Int64")
        df_all["patience"] = pd.to_numeric(df_all["patience"], errors="coerce").astype("Int64")

        df_all["best_epoch"] = pd.to_numeric(df_all["best_epoch"], errors="coerce").astype("Int64")
        df_all["epochs_ran"] = pd.to_numeric(df_all["epochs_ran"], errors="coerce").astype("Int64")

        df_all["dropout"] = pd.to_numeric(df_all["dropout"], errors="coerce")
        df_all["lr"] = pd.to_numeric(df_all["lr"], errors="coerce")
        df_all["w_decay"] = pd.to_numeric(df_all["w_decay"], errors="coerce")
        df_all["best_valid_mse"] = pd.to_numeric(df_all["best_valid_mse"], errors="coerce")
    dt_norm = time.perf_counter() - t0
    if verbose:
        print(f"[{_ts()}] [NORM] dtype window_size/horizon_min/seed/... | dt={dt_norm:.2f}s")

    # expected rows por window_size:
    # n_seeds * n_targets * 2 splits (valid/test)
    expected_rows = int(len(seeds) * len(targets) * 2)

    # 4) Loop por window_size
    for idx, ws in enumerate(window_sizes, start=1):
        t_ws0 = time.perf_counter()

        # 4.1) SKIP check
        t0 = time.perf_counter()
        df_ws = df_all[
            (df_all["model"] == "mlp") &
            (df_all["hidden_dim"] == hidden_dim) &
            (df_all["dropout"] == dropout) &
            (df_all["lr"] == lr) &
            (df_all["w_decay"] == weight_decay) &
            (df_all["batch_size"] == batch_size) &
            (df_all["max_epochs"] == max_epochs) &
            (df_all["patience"] == patience) &
            (df_all["window_size"] == ws) &
            (df_all["target"].isin(list(targets)))
        ]
        dt_filter = time.perf_counter() - t0

        if len(df_ws) >= expected_rows:
            if verbose:
                print(f"[{_ts()}] [{idx}/{len(window_sizes)}] [SKIP] L{ws} | rows={len(df_ws)} "
                      f"(expected>={expected_rows}) | dt_filter={dt_filter:.2f}s")
            continue

        if verbose:
            print("\n" + "=" * 90)
            print(f"[{_ts()}] [{idx}/{len(window_sizes)}] [RUN] MLP incremental | L{ws} | in_dim={ws*n_features} | "
                  f"hd={hidden_dim} | dropout={dropout} | lr={lr} | w_decay={weight_decay} | "
                  f"bs={batch_size} | epochs={max_epochs} | patience={patience} | seeds={list(seeds)}")
            print("=" * 90)

        # 5) RUN
        t0 = time.perf_counter()
        df_new = run_mlp(
            ws,
            n_features=n_features,
            hidden_dim=hidden_dim,
            dropout=dropout,
            lr=lr,
            weight_decay=weight_decay,
            batch_size=batch_size,
            max_epochs=max_epochs,
            patience=patience,
            seeds=list(seeds),
            targets=list(targets),
            verbose=verbose,
        ).copy()
        dt_run = time.perf_counter() - t0

        # agregar hiperparams al DF (por si run_mlp no lo hace)
        df_new["hidden_dim"] = hidden_dim
        df_new["dropout"] = dropout
        df_new["lr"] = lr
        df_new["w_decay"] = weight_decay
        df_new["batch_size"] = batch_size
        df_new["max_epochs"] = max_epochs
        df_new["patience"] = patience

        if verbose:
            print(f"[{_ts()}]   [MLP] run_mlp(L{ws}) OK | new_rows={len(df_new)} | dt={dt_run:.2f}s")

        # 6) Anti-duplicados (sin pisar seeds)
        t0 = time.perf_counter()
        if len(df_all):
            existing_keys = set(tuple(x) for x in df_all[key_cols].dropna().values)
            mask_keep = [tuple(row) not in existing_keys for row in df_new[key_cols].values]
            df_new = df_new.loc[mask_keep].copy()
        dt_dedupe = time.perf_counter() - t0
        if verbose:
            print(f"[{_ts()}]   [DEDUPE] kept_rows={len(df_new)} | dt={dt_dedupe:.2f}s")

        if df_new.empty:
            if verbose:
                print(f"[{_ts()}]   [INFO] L{ws}: no había filas nuevas para agregar.")
            continue

        # 7) Merge + dedupe
        t0 = time.perf_counter()
        df_all = pd.concat([df_all, df_new], ignore_index=True)
        df_all = df_all.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)
        dt_merge = time.perf_counter() - t0
        if verbose:
            print(f"[{_ts()}]   [MERGE] total_rows={len(df_all)} | dt={dt_merge:.2f}s")

        # 8) Save checkpoint
        t0 = time.perf_counter()
        save_seq2one_metrics(df_all, name=name, base_dir=base_dir)
        dt_save = time.perf_counter() - t0
        if verbose:
            print(f"[{_ts()}]   [SAVE] checkpoint OK | dt={dt_save:.2f}s")

        dt_ws = time.perf_counter() - t_ws0
        if verbose:
            print(f"[{_ts()}] [{idx}/{len(window_sizes)}] [DONE] L{ws} | "
                  f"dt_total={dt_ws:.2f}s | filter={dt_filter:.2f}s | run={dt_run:.2f}s | "
                  f"dedupe={dt_dedupe:.2f}s | merge={dt_merge:.2f}s | save={dt_save:.2f}s")

    dt_total = time.perf_counter() - t_total0
    if verbose:
        print(f"\n[{_ts()}] [TOTAL] run_mlp_incremental | dt={dt_total:.2f}s")

    return df_all

In [ ]:
seeds = [
    1, 7, 42, 123, 999,
    2024, 31415, 27182, 8080, 777,
    5555, 8888, 1001, 2025, 9090,
    3333, 4444, 6666, 1212, 2121
]

In [ ]:
df_mlp_all_sizes = run_mlp_incremental(
    window_sizes=[90, 180],
    hidden_dim=128,
    dropout=0.0,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=16384,
    max_epochs=30,
    patience=5,

    # robustez
    seeds =seeds,

    targets=["delta_60"],
    name="mlp",
    verbose=True,
)

[22:40:41] [LOAD] Leyendo métricas existentes (name='mlp') ...
[22:40:41] [LOAD] OK | rows=0 | dt=0.00s
[22:40:41] [NORM] dtype window_size/horizon_min/seed/... | dt=0.00s

[22:40:41] [1/2] [RUN] MLP incremental | L90 | in_dim=3240 | hd=128 | dropout=0.0 | lr=0.001 | w_decay=0.0001 | bs=16384 | epochs=30 | patience=5 | seeds=[1, 7, 42, 123, 999]

[22:40:41] MLP | ROBUSTEZ SEEDS=[1, 7, 42, 123, 999] | WINDOW_SIZE=L90 | in_dim=3240 | hd=128 | dropout=0.0 | lr=0.001 | w_decay=0.0001
[22:40:46] [1/1] [1/5] START target='delta_60' | L90 | seed=1
[22:41:04] [TRAIN] START | in_dim=3240 hidden_dim=128 dropout=0.0 lr=0.001 wd=0.0001 max_epochs=30 patience=5 min_delta=1e-06 device=cuda
[22:41:15] epoch=01 | train_loss=2319.734371 | valid_mse=1912.349900 | BEST | dt_valid=1.70s | dt_epoch=10.55s
[22:41:24] epoch=02 | train_loss=2014.654411 | valid_mse=1783.200575 | BEST | dt_valid=1.53s | dt_epoch=9.35s
[22:41:34] epoch=03 | train_loss=1910.672767 | valid_mse=1710.408152 | BEST | dt_valid=1.60s |

/tmp/ipython-input-3114817817.py:192: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat([df_all, df_new], ignore_index=True)


[23:06:07] [TRAIN] START | in_dim=6480 hidden_dim=128 dropout=0.0 lr=0.001 wd=0.0001 max_epochs=30 patience=5 min_delta=1e-06 device=cuda
[23:06:19] epoch=01 | train_loss=2269.654053 | valid_mse=1831.517372 | BEST | dt_valid=1.93s | dt_epoch=11.47s
[23:06:30] epoch=02 | train_loss=1925.082950 | valid_mse=1682.867346 | BEST | dt_valid=1.96s | dt_epoch=10.90s
[23:06:41] epoch=03 | train_loss=1778.891611 | valid_mse=1607.158783 | BEST | dt_valid=1.98s | dt_epoch=10.88s
[23:06:52] epoch=04 | train_loss=1696.368687 | valid_mse=1524.306673 | BEST | dt_valid=1.96s | dt_epoch=10.97s
[23:07:03] epoch=05 | train_loss=1632.389773 | valid_mse=1571.837971 | no_improve(1/5) | dt_valid=1.89s | dt_epoch=10.96s
[23:07:13] epoch=06 | train_loss=1579.670952 | valid_mse=1501.017287 | BEST | dt_valid=1.90s | dt_epoch=10.99s
[23:07:24] epoch=07 | train_loss=1554.082259 | valid_mse=1529.751310 | no_improve(1/5) | dt_valid=1.89s | dt_epoch=10.99s
[23:07:35] epoch=08 | train_loss=1518.261359 | valid_mse=1402.7

In [ ]:
df_mlp_all_sizes